In [11]:
from utils import get_item_by_id, get_all_training_ids, get_random_training_file

random_well = get_random_training_file()


In [14]:
type_well_data =random_well.typewell

In [13]:
random_well.horizontal_well

,MD,X,Y,Z,ANCC,ASTNU,ASTNL,EGFDU,EGFDL,BUDA,TVT,GR,TVT_input
0,12141.0,3025098.71,1100150.84,-9989.85,-10133.77,-10252.60,-10327.09,-10425.02,-10472.81,-10618.67,11973.95,NaN,11973.95
1,12142.0,3025098.70,1100150.81,-9990.85,-10133.71,-10252.54,-10327.03,-10424.96,-10472.75,-10618.61,11975.01,105.588465,11975.01
2,12143.0,3025098.69,1100150.79,-9991.85,-10133.65,-10252.48,-10326.97,-10424.90,-10472.68,-10618.55,11976.07,108.739259,11976.07
3,12144.0,3025098.68,1100150.76,-9992.85,-10133.58,-10252.41,-10326.90,-10424.83,-10472.62,-10618.48,11977.14,96.104471,11977.14
4,12145.0,3025098.67,1100150.73,-9993.85,-10133.51,-10252.33,-10326.83,-10424.76,-10472.54,-10618.41,11978.21,106.926762,11978.21
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5641,17782.0,3025714.93,1094862.11,-10592.35,-10204.26,-10323.09,-10397.59,-10495.52,-10543.30,-10689.16,12505.96,NaN,NaN
5642,17783.0,3025715.10,1094861.13,-10592.34,-10204.28,-10323.10,-10397.60,-10495.53,-10543.31,-10689.17,12505.94,118.939821,NaN
5643,17784.0,3025715.26,1094860.14,-10592.32,-10204.29,-10323.11,-10397.61,-10495.54,-10543.32,-10689.18,12505.91,111.911128,NaN
5644,17785.0,3025715.42,1094859.15,-10592.31,-10204.30,-10323.12,-10397.62,-10495.55,-10543.33,-10689.19,12505.88,NaN,NaN


In [15]:
type_well_data

,TVT,GR,Geology
0,11961.53,102.67,NaN
1,11962.03,104.18,NaN
2,11962.53,104.77,NaN
3,11963.03,104.75,NaN
4,11963.53,108.87,NaN
...,...,...,...
1379,12650.57,35.35,BUDA
1380,12651.07,35.30,BUDA
1381,12651.57,34.91,BUDA
1382,12652.07,35.11,BUDA


In [16]:
import copy
import random
from typing import Literal

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

from training import (
    RNN_DROP_COLUMNS,
    RNN_FEATURE_COLUMNS,
    StatefulTVTWindowDataset,
    prepare_rnn_features,
)
from utils import (
    get_all_test_ids,
    get_all_training_ids,
    test_folder,
    train_folder,
)

SEED = 42
VALID_SIZE = 0.15
WINDOW_SIZE = 256
CONTEXT_LENGTH = 256
MIN_FORECAST_LENGTH = 256
MIN_FORECAST_FRACTION = 0.40
REAL_PS_ANCHOR_PROB = 0.70
REAL_PS_FRACTION_RANGE = (0.20, 0.35)
ANCHORS_PER_WELL = 4
EPISODE_BATCH_SIZE = 32
NUM_EPOCHS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


/Users/doruksarigun/Kaggle/ROGII - Wellbore Geology Prediction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
all_training_ids = get_all_training_ids()
reserved_test_ids = set(get_all_test_ids())
modeling_ids = sorted(set(all_training_ids) - reserved_test_ids)

train_well_ids, valid_well_ids = train_test_split(
    modeling_ids,
    test_size=VALID_SIZE,
    random_state=SEED,
    shuffle=True,
)

assert set(train_well_ids).isdisjoint(valid_well_ids)
assert set(train_well_ids).isdisjoint(reserved_test_ids)
print(f"Train wells: {len(train_well_ids)}")
print(f"Validation wells: {len(valid_well_ids)}")
print(f"Reserved test wells: {len(reserved_test_ids)}")

In [ ]:
train_wells = prepare_rnn_features(
    training_ids=train_well_ids,
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
)
valid_wells = prepare_rnn_features(
    training_ids=valid_well_ids,
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
)
test_wells = prepare_rnn_features(
    training_ids=sorted(reserved_test_ids),
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
    data_dir=test_folder,
    target_data_dir=train_folder,
)

len(train_wells), len(valid_wells), len(test_wells)

In [ ]:
TYPEWELL_GRID_POINTS = 512
TYPEWELL_TOKEN_HALF_WINDOW = 6  # token = 13 komsu GR + dtvt


def build_typewell_grid(
    typewell_df: pd.DataFrame,
    num_points: int = TYPEWELL_GRID_POINTS,
) -> tuple[np.ndarray, np.ndarray]:
    """Typewell'i uniform TVT gridine resample eder.

    Typewell uzunluklari kuyudan kuyuya degisir; model sabit boyutlu
    bellek bekler. Uniform grid + lineer interpolasyon profili korur.
    """
    typewell = (
        typewell_df.dropna(subset=["TVT", "GR"])
        .sort_values("TVT")
        .drop_duplicates("TVT")
    )
    if len(typewell) < 2:
        raise ValueError("Typewell icinde yeterli TVT-GR verisi yok.")

    tvt_grid = np.linspace(
        typewell["TVT"].min(),
        typewell["TVT"].max(),
        num_points,
    ).astype(np.float32)
    gr_grid = np.interp(
        tvt_grid,
        typewell["TVT"].to_numpy(dtype=float),
        typewell["GR"].to_numpy(dtype=float),
    ).astype(np.float32)
    return tvt_grid, gr_grid


def fit_typewell_gr_calibration(
    well_data: pd.DataFrame,
    tvt_grid: np.ndarray,
    gr_grid: np.ndarray,
) -> tuple[float, float]:
    """PS oncesi bilinen (TVT, GR) ciftleriyle typewell→horizontal GR
    olcek donusumunu fit eder.

    Yalnizca gercekten olculmus GR satirlari kullanilir (GR_isna == 0);
    impute edilmis degerler kalibrasyonu kirletmesin. observed_TVT yalnizca
    PS oncesinde dolu oldugu icin leakage olusmaz.
    """
    known = well_data.loc[
        well_data["observed_TVT"].notna() & (well_data["GR_isna"] == 0),
        ["observed_TVT", "GR"],
    ]
    if len(known) < 3:
        return 1.0, 0.0

    expected_gr = np.interp(
        known["observed_TVT"].to_numpy(dtype=float), tvt_grid, gr_grid
    )
    observed_gr = known["GR"].to_numpy(dtype=float)
    scale, offset = np.polyfit(expected_gr, observed_gr, 1)
    if not np.isfinite(scale) or abs(scale) < 1e-8:
        return 1.0, 0.0
    return float(scale), float(offset)


def build_typewell_gr_tokens(
    gr_grid: np.ndarray,
    half_window: int = TYPEWELL_TOKEN_HALF_WINDOW,
) -> np.ndarray:
    """Her grid noktasi icin komsu GR penceresinden token uretir.

    Tek GR degeri stratigrafide belirsizdir (ayni deger bircok derinlikte
    gorulur); pattern penceresi eslesmeyi tekillestirir. Kenarlar edge-pad
    ile korunur. Cikti: (num_points, 2*half_window + 1).
    """
    padded = np.pad(gr_grid, half_window, mode="edge")
    return np.lib.stride_tricks.sliding_window_view(
        padded, 2 * half_window + 1
    ).astype(np.float32)


# --- sanity check: rastgele bir train kuyusu ---
_example_id = train_well_ids[0]
_example_well = train_wells[_example_id]
_tvt_grid, _gr_grid = build_typewell_grid(_example_well.typewell_df)
_scale, _offset = fit_typewell_gr_calibration(
    _example_well.data, _tvt_grid, _gr_grid
)
_tokens = build_typewell_gr_tokens(_gr_grid * _scale + _offset)

assert _tvt_grid.shape == _gr_grid.shape == (TYPEWELL_GRID_POINTS,)
assert np.all(np.diff(_tvt_grid) > 0)
assert np.isfinite(_gr_grid).all()
assert _tokens.shape == (TYPEWELL_GRID_POINTS, 2 * TYPEWELL_TOKEN_HALF_WINDOW + 1)
assert np.isfinite(_tokens).all()
assert np.allclose(_tokens[:, TYPEWELL_TOKEN_HALF_WINDOW], _gr_grid * _scale + _offset)

print(f"Example well: {_example_id}")
print(f"tvt_grid: [{_tvt_grid.min():.1f}, {_tvt_grid.max():.1f}] ({len(_tvt_grid)} pts)")
print(f"GR calibration: scale={_scale:.3f}, offset={_offset:.2f}")
print(f"tokens: {_tokens.shape}")

In [ ]:
def attach_typewell_memory(prepared_wells):
    """Her kuyuya typewell bellegini kurar ve TrainingData uzerine cache'ler.

    Kuyu basina bir kez hesaplanir; tum segment ve episode'lar ayni
    referansi kullanir. dtvt anchor'a bagli oldugu icin burada DEGIL,
    dataset kurulurken (episode basina) hesaplanir.
    """
    for well in tqdm(prepared_wells.values(), desc="Typewell memory", unit="well"):
        tvt_grid, gr_grid = build_typewell_grid(well.typewell_df)
        scale, offset = fit_typewell_gr_calibration(
            well.data, tvt_grid, gr_grid
        )
        well.tw_tvt_grid = tvt_grid
        well.tw_gr_tokens = build_typewell_gr_tokens(
            gr_grid * scale + offset
        )
        well.tw_gr_calibration = (scale, offset)


attach_typewell_memory(train_wells)
attach_typewell_memory(valid_wells)
attach_typewell_memory(test_wells)

_scales = [w.tw_gr_calibration[0] for w in train_wells.values()]
print(f"GR calibration scale: median={np.median(_scales):.3f}, "
      f"p5={np.percentile(_scales, 5):.3f}, p95={np.percentile(_scales, 95):.3f}")

In [ ]:
MASK_FEATURE_COLUMNS = [
    "GR_isna",
    "GR_typewell_imputed_mask",
    "GR_neighbor_imputed_mask",
]
CONTINUOUS_FEATURE_COLUMNS = [
    column for column in RNN_FEATURE_COLUMNS
    if column not in MASK_FEATURE_COLUMNS
]

feature_scaler = StandardScaler()
target_scaler = StandardScaler()
for well in train_wells.values():
    feature_scaler.partial_fit(well.data[CONTINUOUS_FEATURE_COLUMNS])
    target_scaler.partial_fit(well.data[["TVT"]])

target_mean = float(target_scaler.mean_[0])
target_scale = float(target_scaler.scale_[0])

# Typewell tokenlari horizontal GR kanaliyla ayni istatistikle olceklenir;
# boylece attention iki tarafta ayni birimleri karsilastirir.
_gr_channel = CONTINUOUS_FEATURE_COLUMNS.index("GR")
gr_mean = float(feature_scaler.mean_[_gr_channel])
gr_scale = float(feature_scaler.scale_[_gr_channel])


def apply_scalers(prepared_wells):
    for well in prepared_wells.values():
        well.data.loc[:, CONTINUOUS_FEATURE_COLUMNS] = (
            feature_scaler.transform(
                well.data[CONTINUOUS_FEATURE_COLUMNS]
            )
        )
        well.data.loc[:, "TVT_scaled"] = target_scaler.transform(
            well.data[["TVT"]]
        ).ravel()
        well.data.loc[:, "observed_TVT_scaled"] = 0.0
        observed_mask = well.data["observed_TVT"].notna()
        well.data.loc[observed_mask, "observed_TVT_scaled"] = (
            well.data.loc[observed_mask, "observed_TVT"] - target_mean
        ) / target_scale
        well.tw_gr_tokens_scaled = (
            (well.tw_gr_tokens - gr_mean) / gr_scale
        ).astype(np.float32)


apply_scalers(train_wells)
apply_scalers(valid_wells)
apply_scalers(test_wells)
target_mean, target_scale

In [ ]:
class TypewellStatefulTVTWindowDataset(StatefulTVTWindowDataset):
    """Stateful segment akisina kuyunun typewell bellegini ekler.

    tw_tokens = [13 komsu GR (scaled), dtvt] -> (M, 14). dtvt, anchor'in
    (PS'ten onceki son satir) ham TVT'sine gore relatif ve target_scale ile
    olceklidir; boylece soft pointer ciktisi TVT_scaled ile ayni birimdedir.
    Segmentler ayni tensor referansini paylasir, kopya yapilmaz.
    """

    def __init__(
        self,
        *,
        tw_tvt_grid: np.ndarray,
        tw_gr_tokens_scaled: np.ndarray,
        target_scale: float,
        **kwargs,
    ):
        super().__init__(**kwargs)

        anchor_row = self.prediction_start - 1
        anchor_tvt = float(self.data.loc[anchor_row, "TVT"])
        dtvt = (
            (tw_tvt_grid.astype(np.float64) - anchor_tvt) / target_scale
        ).astype(np.float32)

        self.anchor_tvt = anchor_tvt
        self.tw_dtvt = torch.from_numpy(dtvt)
        self.tw_tokens = torch.from_numpy(
            np.concatenate(
                [tw_gr_tokens_scaled, dtvt[:, None]], axis=1
            ).astype(np.float32)
        )

    def __getitem__(self, idx):
        item = super().__getitem__(idx)
        item["tw_tokens"] = self.tw_tokens
        item["tw_dtvt"] = self.tw_dtvt
        return item


def make_real_datasets(prepared_wells):
    return {
        well_id: TypewellStatefulTVTWindowDataset(
            data=well.data,
            feature_columns=RNN_FEATURE_COLUMNS,
            target_column="TVT_scaled",
            window_size=WINDOW_SIZE,
            context_length=CONTEXT_LENGTH,
            mode="real",
            prediction_mask_column="is_prediction_row",
            observed_context_column="observed_TVT_scaled",
            tw_tvt_grid=well.tw_tvt_grid,
            tw_gr_tokens_scaled=well.tw_gr_tokens_scaled,
            target_scale=target_scale,
        )
        for well_id, well in prepared_wells.items()
    }


valid_datasets = make_real_datasets(valid_wells)
test_datasets = make_real_datasets(test_wells)

# --- sanity checks ---
_ds = next(iter(test_datasets.values()))
_seg = _ds[0]

assert _seg["tw_tokens"].shape == (
    TYPEWELL_GRID_POINTS, 2 * TYPEWELL_TOKEN_HALF_WINDOW + 2
)
assert _seg["tw_dtvt"].shape == (TYPEWELL_GRID_POINTS,)
assert torch.isfinite(_seg["tw_tokens"]).all()
assert torch.allclose(_seg["tw_tokens"][:, -1], _seg["tw_dtvt"])
# anchor typewell araliginin icinde olmali -> dtvt isaret degistirir
assert (_seg["tw_dtvt"] < 0).any() and (_seg["tw_dtvt"] > 0).any(), (
    "anchor typewell TVT araliginin disinda"
)
# segmentler ayni references paylasmali (bellek kontrolu)
assert _ds[0]["tw_tokens"] is _ds[len(_ds) - 1]["tw_tokens"]

print(f"tw_tokens: {tuple(_seg['tw_tokens'].shape)}")
print(f"anchor TVT: {_ds.anchor_tvt:.2f}")
print(f"dtvt range (scaled): [{_seg['tw_dtvt'].min():.3f}, {_seg['tw_dtvt'].max():.3f}]")
print(f"features per segment: {tuple(_seg['features'].shape)}")
print("Dataset hazir — sonraki adim: cross-attention'li model.")

In [ ]:
class TypewellAttnTVTModel(nn.Module):
    """GRU horizontal akisi ozetler; cross-attention typewell'i sorgular.

    Her timestep'te GRU hidden'i query olur, 512 typewell tokeni key/value
    olur. alpha typewell derinligi uzerinde bir dagilimdir; soft_dtvt bu
    dagilimin beklenen TVT delta'sidir (pointer tahmini). Head, GRU ozeti +
    haritadan okunan baglam + pointer'i birlestirir.
    """

    def __init__(
        self,
        input_size: int,
        tw_channels: int = 14,
        hidden_size: int = 64,
        num_heads: int = 4,
        n_layers: int = 2,
        dropout: float = 0.15,
    ):
        super().__init__()
        self.rnn_layer = nn.GRU(
            input_size,
            hidden_size,
            n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.typewell_proj = nn.Linear(tw_channels, hidden_size)
        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.attn = nn.MultiheadAttention(
            hidden_size,
            num_heads,
            batch_first=True,
            dropout=dropout,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size * 2 + 1),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2 + 1, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x, tw_tokens, tw_dtvt, hidden=None):
        # x: (B, T, F), tw_tokens: (B, M, C), tw_dtvt: (B, M)
        rnn_out, hidden = self.rnn_layer(x, hidden)      # (B, T, H)

        query = self.q_proj(rnn_out)                     # (B, T, H)
        memory = self.typewell_proj(tw_tokens)           # (B, M, H)
        context, alpha = self.attn(
            query,
            memory,
            memory,
            need_weights=True,
            average_attn_weights=True,
        )                                                # (B,T,H), (B,T,M)

        soft_dtvt = torch.einsum("btm,bm->bt", alpha, tw_dtvt)  # (B, T)

        head_input = torch.cat(
            [rnn_out, context, soft_dtvt.unsqueeze(-1)],
            dim=-1,
        )
        predictions = self.head(head_input).squeeze(-1)  # (B, T)
        return predictions, hidden, alpha

In [ ]:
model = TypewellAttnTVTModel(
    input_size=len(RNN_FEATURE_COLUMNS) + 2,
    tw_channels=2 * TYPEWELL_TOKEN_HALF_WINDOW + 2,
    hidden_size=64,
    num_heads=4,
    n_layers=2,
    dropout=0.15,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-3, weight_decay=1e-5
)

# --- smoke test: tek segment ileri gecis ---
# eval(): attention dropout'u softmax sonrasi uygulanir; train modunda
# alpha satir toplami 1 olmaz ve asagidaki kontrol yaniltici sekilde patlar.
model.eval()
_ds = next(iter(test_datasets.values()))
_seg = _ds[0]
with torch.no_grad():
    _pred, _hidden, _alpha = model(
        _seg["features"].unsqueeze(0).to(device),
        _seg["tw_tokens"].unsqueeze(0).to(device),
        _seg["tw_dtvt"].unsqueeze(0).to(device),
    )

assert _pred.shape == _seg["target"].unsqueeze(0).shape
assert _alpha.shape == (1, _seg["features"].shape[0], TYPEWELL_GRID_POINTS)
assert torch.allclose(
    _alpha.sum(-1), torch.ones_like(_alpha.sum(-1)), atol=1e-5
)
n_params = sum(p.numel() for p in model.parameters())
print(f"pred: {tuple(_pred.shape)}, alpha: {tuple(_alpha.shape)}")
print(f"parameters: {n_params:,}")

In [ ]:
def detach_hidden(hidden):
    if isinstance(hidden, tuple):
        return tuple(state.detach() for state in hidden)
    return hidden.detach()


def masked_mse_loss(predictions, targets, loss_mask):
    return ((predictions - targets) ** 2)[loss_mask].mean()


def sample_prediction_start(well, rng):
    """Sentetik PS'i gercek PS dagilimina agirlikli olarak orneklar."""
    num_rows = len(well.data)
    min_forecast = max(
        MIN_FORECAST_LENGTH,
        int(num_rows * MIN_FORECAST_FRACTION),
    )
    first = CONTEXT_LENGTH
    last = num_rows - min_forecast
    if last < first:
        return None

    if rng.random() < REAL_PS_ANCHOR_PROB:
        low = max(first, int(num_rows * REAL_PS_FRACTION_RANGE[0]))
        high = min(last, int(num_rows * REAL_PS_FRACTION_RANGE[1]))
        if low <= high:
            return rng.randint(low, high)

    return rng.randint(first, last)


def make_batch_segments(max_prefix, total_len):
    """Batch-ortak segment sinirlari.

    Tum episode'lar PS - CONTEXT_LENGTH noktasinda hizalandigi icin ayni
    sinirlar herkes icin gecerlidir: prefix warmup dilimleri, ardindan
    context + window'luk ilk loss dilimi, sonra sabit window dilimleri.
    """
    segments = []
    for start in range(0, max_prefix, WINDOW_SIZE):
        segments.append((start, min(start + WINDOW_SIZE, max_prefix)))
    first_end = min(total_len, max_prefix + CONTEXT_LENGTH + WINDOW_SIZE)
    segments.append((max_prefix, first_end))
    for start in range(first_end, total_len, WINDOW_SIZE):
        segments.append((start, min(start + WINDOW_SIZE, total_len)))
    return segments


def build_episode_batch(episode_datasets):
    """N episode'u PS hizali tek batch'e paketler.

    Prefix soldan pad edilir (herkes ayni kolonda context'e girer), forecast
    kuyruklari sagdan pad edilir (loss_mask=False oldugu icin zararsiz).
    Pad edilen satirlar sifir feature tasir; warmup hidden'i gercek veriyle
    uzerine yazilir, kuyruk pad'leri ise hic loss uretmez.
    """
    prefix_lens = [
        max(0, ds.prediction_start - CONTEXT_LENGTH)
        for ds in episode_datasets
    ]
    main_lens = [
        len(ds.features) - prefix
        for ds, prefix in zip(episode_datasets, prefix_lens)
    ]
    max_prefix = max(prefix_lens)
    max_main = max(main_lens)
    total_len = max_prefix + max_main

    batch_size = len(episode_datasets)
    feature_dim = episode_datasets[0].features.shape[1]
    features = torch.zeros(batch_size, total_len, feature_dim)
    targets = torch.zeros(batch_size, total_len)
    loss_mask = torch.zeros(batch_size, total_len, dtype=torch.bool)

    for i, (ds, prefix) in enumerate(zip(episode_datasets, prefix_lens)):
        offset = max_prefix - prefix
        rows = len(ds.features)
        features[i, offset : offset + rows] = ds.features
        targets[i, offset : offset + rows] = ds.target
        loss_mask[i, offset : offset + rows] = ds.loss_mask

    return {
        "features": features,
        "targets": targets,
        "loss_mask": loss_mask,
        "tw_tokens": torch.stack([ds.tw_tokens for ds in episode_datasets]),
        "tw_dtvt": torch.stack([ds.tw_dtvt for ds in episode_datasets]),
        "segments": make_batch_segments(max_prefix, total_len),
    }


def run_batch(model, batch, optimizer=None):
    """Bir episode batch'ini segment segment isler; hidden state tasir.

    optimizer verilirse loss'lu segmentlerde backprop yapilir (train),
    verilmezse yalnizca skor toplanir (eval). Donus: (sq_err_sum, n_obs).
    """
    features = batch["features"].to(device)
    targets = batch["targets"].to(device)
    loss_mask = batch["loss_mask"].to(device)
    tw_tokens = batch["tw_tokens"].to(device)
    tw_dtvt = batch["tw_dtvt"].to(device)

    squared_error_sum = 0.0
    observation_count = 0
    hidden = None

    for start, end in batch["segments"]:
        x = features[:, start:end]
        y = targets[:, start:end]
        mask = loss_mask[:, start:end]

        if optimizer is not None and not mask.any():
            with torch.no_grad():
                _, hidden, _ = model(x, tw_tokens, tw_dtvt, hidden)
            hidden = detach_hidden(hidden)
            continue

        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
            predictions, hidden, _ = model(x, tw_tokens, tw_dtvt, hidden)
            loss = masked_mse_loss(predictions, y, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            predictions = predictions.detach()
        else:
            predictions, hidden, _ = model(x, tw_tokens, tw_dtvt, hidden)
        hidden = detach_hidden(hidden)

        if mask.any():
            errors_tvt = (predictions - y) * target_scale
            squared_error_sum += torch.sum(errors_tvt[mask] ** 2).item()
            observation_count += int(mask.sum().item())

    return squared_error_sum, observation_count


def make_episode_dataset(well, prediction_start):
    return TypewellStatefulTVTWindowDataset(
        data=well.data,
        feature_columns=RNN_FEATURE_COLUMNS,
        target_column="TVT_scaled",
        window_size=WINDOW_SIZE,
        context_length=CONTEXT_LENGTH,
        mode="synthetic",
        prediction_start=prediction_start,
        tw_tvt_grid=well.tw_tvt_grid,
        tw_gr_tokens_scaled=well.tw_gr_tokens_scaled,
        target_scale=target_scale,
    )


def train_one_epoch(model, prepared_wells, optimizer, rng, epoch=None):
    model.train()

    # Episode'lari topla; benzer uzunluklari ayni batch'e koy (pad israfini
    # azaltir), batch sirasini karistir.
    episodes = []
    for well_id, well in prepared_wells.items():
        for _ in range(ANCHORS_PER_WELL):
            prediction_start = sample_prediction_start(well, rng)
            if prediction_start is not None:
                episodes.append((well_id, prediction_start))
    episodes.sort(key=lambda item: len(prepared_wells[item[0]].data))
    batches = [
        episodes[i : i + EPISODE_BATCH_SIZE]
        for i in range(0, len(episodes), EPISODE_BATCH_SIZE)
    ]
    rng.shuffle(batches)

    squared_error_sum = 0.0
    observation_count = 0
    progress = tqdm(
        batches,
        desc=f"Epoch {epoch}" if epoch is not None else "Train",
        unit="batch",
        leave=False,
    )
    for batch_episodes in progress:
        episode_datasets = [
            make_episode_dataset(prepared_wells[well_id], prediction_start)
            for well_id, prediction_start in batch_episodes
        ]
        batch = build_episode_batch(episode_datasets)
        sq_err, n_obs = run_batch(model, batch, optimizer)
        squared_error_sum += sq_err
        observation_count += n_obs

        if observation_count > 0:
            progress.set_postfix(
                running_rmse=f"{np.sqrt(squared_error_sum / observation_count):.4f}"
            )

    if observation_count == 0:
        raise ValueError("Training epoch'unda loss satiri bulunamadi.")
    return float(np.sqrt(squared_error_sum / observation_count))


@torch.no_grad()
def evaluate_real_ps(model, datasets):
    model.eval()

    ordered = sorted(
        datasets.values(), key=lambda ds: len(ds.features)
    )
    batches = [
        ordered[i : i + EPISODE_BATCH_SIZE]
        for i in range(0, len(ordered), EPISODE_BATCH_SIZE)
    ]

    squared_error_sum = 0.0
    observation_count = 0
    for batch_datasets in tqdm(
        batches, desc="Eval", unit="batch", leave=False
    ):
        batch = build_episode_batch(batch_datasets)
        sq_err, n_obs = run_batch(model, batch, optimizer=None)
        squared_error_sum += sq_err
        observation_count += n_obs

    if observation_count == 0:
        raise ValueError("Gercek PS sonrasinda skorlanacak satir yok.")
    return float(np.sqrt(squared_error_sum / observation_count))

In [ ]:
history = []
best_valid_rmse = float("inf")
best_state = None
anchor_rng = random.Random(SEED)

for epoch in range(1, NUM_EPOCHS + 1):
    train_rmse = train_one_epoch(
        model, train_wells, optimizer, anchor_rng, epoch=epoch
    )
    valid_rmse = evaluate_real_ps(model, valid_datasets)
    history.append({
        "epoch": epoch,
        "synthetic_train_rmse": train_rmse,
        "real_ps_valid_rmse": valid_rmse,
    })
    if valid_rmse < best_valid_rmse:
        best_valid_rmse = valid_rmse
        best_state = copy.deepcopy(model.state_dict())
    print(
        f"Epoch {epoch:02d} | synthetic train RMSE: {train_rmse:.4f} | "
        f"real-PS valid RMSE: {valid_rmse:.4f}"
    )

if best_state is not None:
    model.load_state_dict(best_state)
test_rmse = evaluate_real_ps(model, test_datasets)
print(f"Typewell-attention reserved test RMSE: {test_rmse:.4f}")
pd.DataFrame(history)

In [ ]:
import matplotlib.pyplot as plt

# Attention haritasi: model typewell'in dogru bolgesine mi bakiyor?
# Satirlar zaman (MD sirasi), sutunlar typewell derinligi. Beklenen goruntu:
# gercek TVT cizgisini takip eden koyu bir serit.


@torch.no_grad()
def collect_attention_map(model, dataset):
    model.eval()
    hidden = None
    tw_tokens = dataset.tw_tokens.unsqueeze(0).to(device)
    tw_dtvt = dataset.tw_dtvt.unsqueeze(0).to(device)

    alphas, preds = [], []
    for segment in dataset:
        features = segment["features"].unsqueeze(0).to(device)
        predictions, hidden, alpha = model(
            features, tw_tokens, tw_dtvt, hidden
        )
        alphas.append(alpha.squeeze(0).cpu())
        preds.append(predictions.squeeze(0).cpu())
    return torch.cat(alphas), torch.cat(preds)


_well_id = max(
    test_datasets,
    key=lambda wid: int(test_wells[wid].data["is_prediction_row"].sum()),
)
_ds = test_datasets[_well_id]
_alpha_map, _preds = collect_attention_map(model, _ds)

_well = test_wells[_well_id]
_true_tvt = _well.data["TVT"].to_numpy()
_pred_tvt = _preds.numpy() * target_scale + target_mean

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].imshow(
    _alpha_map.numpy().T,
    aspect="auto",
    origin="lower",
    extent=[0, len(_alpha_map), _well.tw_tvt_grid[0], _well.tw_tvt_grid[-1]],
    cmap="viridis",
)
axes[0].plot(np.arange(len(_true_tvt)), _true_tvt, "r-", lw=0.8, label="true TVT")
axes[0].axvline(_ds.prediction_start, color="w", ls="--", lw=1, label="PS")
axes[0].set_ylabel("typewell TVT")
axes[0].set_title(f"Attention map — {_well_id}")
axes[0].legend(loc="upper right")

axes[1].plot(_true_tvt, label="true TVT", lw=1)
axes[1].plot(_pred_tvt, label="predicted TVT", lw=1)
axes[1].axvline(_ds.prediction_start, color="k", ls="--", lw=1, label="PS")
axes[1].set_xlabel("row (MD sirasi)")
axes[1].set_ylabel("TVT")
axes[1].legend()

plt.tight_layout()
plt.show()

_pred_region = slice(_ds.prediction_start, len(_true_tvt))
_rmse = float(np.sqrt(np.mean(
    (_pred_tvt[_pred_region] - _true_tvt[_pred_region]) ** 2
)))
print(f"{_well_id} prediction-region RMSE: {_rmse:.3f}")